# Prediction

Load the pretrained classifier and inspect what it predicts. No training happens here — the model in `artifacts/models/` is served as-is.

In [ ]:
import sys
from pathlib import Path

BACKEND_ROOT = Path.cwd().parents[2]
sys.path[:0] = [str(BACKEND_ROOT), str(BACKEND_ROOT / "src")]

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

from prayaas.config.configuration import CLASS_LABELS, settings

CLASS_LABELS

## Load the model

The architecture was serialised with Keras 2.15, which Keras 3 cannot deserialise (`Could not locate class 'Functional'`). `pipeline/model_loader.py` uses `tf_keras` — the Keras 2 API — which loads it correctly.

In [ ]:
from pipeline.model_loader import load_model

model = load_model()
print("input :", model.input_shape)
print("output:", model.output_shape)
print("params:", f"{model.count_params():,}")
print("layers:", len(model.layers))

In [ ]:
# Final layers — the classification head sitting on the backbone.
for layer in model.layers[-4:]:
    print(f"{layer.name:30s} {layer.__class__.__name__:24s} -> {layer.output_shape}")

## Predict a single image

In [ ]:
from pipeline.data_ingestion import DataIngestion
from pipeline.prediction_pipeline import PredictionPipeline

index = DataIngestion().run()
pipeline = PredictionPipeline()

sample_path = index[index["folder"] == "CANCER"]["path"].iloc[0]
result = pipeline.predict_bytes(Path(sample_path).read_bytes())
result

In [ ]:
with Image.open(sample_path) as im:
    plt.imshow(im)
plt.axis("off")
plt.title(f"{result['label']} ({result['confidence']:.1%})")
plt.show()

## Predict a stratified batch

In [ ]:
sample = index[index.groupby("label").cumcount() < 8].reset_index(drop=True)
predictions = pipeline.predict_paths(sample["path"].tolist())

pd.DataFrame({
    "folder": sample["folder"],
    "predicted": [p["label"] for p in predictions],
    "confidence": [p["confidence"] for p in predictions],
})

## Visualise predictions

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))

for ax, (_, row), prediction in zip(axes.ravel(), sample.iterrows(), predictions):
    with Image.open(row["path"]) as im:
        ax.imshow(im)
    ax.set_title(
        f"true: {row['folder']}\npred: {prediction['label']} ({prediction['confidence']:.0%})",
        fontsize=8,
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

## Does CLAHE actually help?

`APPLY_CLAHE` defaults to true because the original client enhanced images before upload. This compares mean confidence in the correct class with and without it.

In [ ]:
from pipeline.data_preprocessing import DataPreprocessing, apply_clahe

paths = sample["path"].tolist()
truth = sample["label"].tolist()

def confidence_in_truth(use_clahe):
    batch = []
    for path in paths:
        with Image.open(path) as im:
            array = np.array(im.convert("RGB").resize((224, 224)))
        if use_clahe:
            array = apply_clahe(array)
        batch.append(array.astype(np.float32) / 255.0)

    probabilities = pipeline.predict_array(np.stack(batch))
    return float(np.mean([p[t] for p, t in zip(probabilities, truth)]))

print(f"with CLAHE:    {confidence_in_truth(True):.4f}")
print(f"without CLAHE: {confidence_in_truth(False):.4f}")
print("\n(Higher = more confident in the correct class.)")